# PIG Inverse Problem Walkthrough

This notebook explains the pipeline *backwards*.

Instead of starting from model internals and ending with a graph, we start from the question:

> Given a small observed patch-effect matrix, what hidden mechanism are we trying to recover?

In PIG, the inverse problem is:

$$
\text{Given } \mathbf{X} \in \mathbb{R}^{N \times D}, \text{ infer a sparse graph } G
\text{ that summarizes stable structure in those patch effects.}
$$

Here:

- $N$ = number of prompt pairs / examples
- $D$ = number of patch nodes `(layer, token, component)`
- $X_{i,j}$ = effect of patching node $j$ on example $i$

We will keep everything tiny and explicit: 4 nodes, 6 examples, hand-made matrices, and visual experiments.


## TL;DR

The pipeline is trying to answer three nested questions.

1. **Node level**: which patch locations consistently matter for the target behavior?
2. **Graph level**: which nodes move together across examples, suggesting a shared mechanism?
3. **Slice level**: which prompt slices induce similar graphs, suggesting similar circuits?

The hard part is that the map from hidden mechanism to observed patch-effect matrix is many-to-one.
That is why this is an inverse problem rather than a direct readout.


## Forward problem vs inverse problem

The forward object is a single patch effect:

$$
E_u^{(i)} = O(\text{patch}(x_{\mathrm{crp}}^{(i)}; u)) - O(x_{\mathrm{crp}}^{(i)})
$$

For one example $i$, stacking over nodes $u = 1, \dots, D$ gives a row vector
$\mathbf{x}^{(i)} \in \mathbb{R}^D$.
Stacking over examples gives the matrix:

$$
\mathbf{X} =
\begin{bmatrix}
- (\mathbf{x}^{(1)})^\top - \\
- (\mathbf{x}^{(2)})^\top - \\
\vdots \\
- (\mathbf{x}^{(N)})^\top -
\end{bmatrix}
\in \mathbb{R}^{N \times D}
$$

A useful toy generative view is

$$
\mathbf{x}^{(i)} = A \mathbf{z}^{(i)} + \varepsilon^{(i)}
$$

where $\mathbf{z}^{(i)}$ are latent circuit factors and $A$ says which nodes respond to them.

The inverse problem is: from only $\mathbf{X}$, recover a graph that captures the stable dependency pattern.
We are **not** directly recovering the true transformer weights or a unique causal DAG.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pig.causal import (
    activation_influence_score,
    mediation_score,
    necessity_score,
    restoration_fraction,
)
from pig.embeddings import compute_wl_features
from pig.graph import GraphBuilder
from pig.patching import ComponentSpec, PatchEffectDataset, PatchEffectTensor
from pig.prompts import PromptPair, SliceLabel

plt.rcParams["figure.dpi"] = 130
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

node_labels = ["L0T0", "L0T1", "L1T0", "L1T1"]
layer_positions = {
    "L0T0": (0.0, 1.0),
    "L0T1": (1.0, 1.0),
    "L1T0": (0.2, 0.0),
    "L1T1": (1.2, 0.0),
}


def show_matrix(
    title,
    matrix,
    row_labels,
    col_labels,
    ax=None,
    cmap="RdBu_r",
    vmin=None,
    vmax=None,
    annotate=None,
    text_size=8,
):
    if ax is None:
        _, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(matrix, cmap=cmap, vmin=vmin, vmax=vmax, aspect="auto")
    ax.set_title(title)
    ax.set_xticks(range(len(col_labels)))
    ax.set_xticklabels(col_labels, rotation=45, ha="right")
    ax.set_yticks(range(len(row_labels)))
    ax.set_yticklabels(row_labels)
    if annotate is None:
        annotate = matrix.size <= 64
    if annotate:
        for i in range(matrix.shape[0]):
            for j in range(matrix.shape[1]):
                ax.text(j, i, f"{matrix[i, j]:+.2f}", ha="center", va="center", fontsize=text_size)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    return ax


def show_profiles(matrix, labels, ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 3))
    x = np.arange(matrix.shape[0])
    for j, label in enumerate(labels):
        ax.plot(x, matrix[:, j], marker="o", linewidth=2, label=label)
    ax.axhline(0.0, color="black", linewidth=1, alpha=0.4)
    ax.set_xticks(x)
    ax.set_xticklabels([f"ex{i}" for i in x])
    ax.set_ylabel("effect")
    ax.set_title("Per-node effect profiles across examples")
    ax.legend(loc="upper right")
    return ax


def show_scatter(x, y, x_label, y_label, ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(3.8, 3.5))
    ax.scatter(x, y, s=55)
    for i, (xi, yi) in enumerate(zip(x, y)):
        ax.text(xi + 0.03, yi + 0.03, f"ex{i}", fontsize=8)
    ax.axhline(0.0, color="black", linewidth=1, alpha=0.4)
    ax.axvline(0.0, color="black", linewidth=1, alpha=0.4)
    ax.set_xlabel(x_label)
    ax.set_ylabel(y_label)
    ax.set_title(f"{x_label} vs {y_label}")
    return ax


def correlation_matrix(matrix):
    matrix = np.asarray(matrix, dtype=np.float32)
    centered = matrix - matrix.mean(axis=0, keepdims=True)
    std = np.std(centered, axis=0, keepdims=True)
    std = np.where(std < 1e-8, 1.0, std)
    normalized = centered / std
    corr = (normalized.T @ normalized) / matrix.shape[0]
    np.fill_diagonal(corr, 0.0)
    return corr


def make_dataset(matrix, slice_label):
    dataset = PatchEffectDataset()
    component_axis = [ComponentSpec(node_type="res")]
    for i, row in enumerate(matrix):
        pair = PromptPair(
            x_cln=f"clean-{i}",
            x_crp=f"corrupted-{i}",
            y_star="target",
            slice_label=slice_label,
            meta={},
        )
        tensor = PatchEffectTensor(
            effects=np.asarray(row, dtype=np.float32).reshape(2, 2, 1),
            component_axis=component_axis,
            prompt_pair=pair,
            base_score=0.0,
            clean_score=1.0,
            token_labels=["tok0", "tok1"],
        )
        dataset.add(tensor)
    return dataset


def draw_graph(graph, ax, title):
    ax.set_title(title)
    for label, (x, y) in layer_positions.items():
        ax.scatter(x, y, s=900, color="#f6d365", edgecolor="black", zorder=3)
        ax.text(x, y, label, ha="center", va="center", fontsize=9, weight="bold")
    for edge in graph.edges:
        src = graph.nodes[edge.src]
        dst = graph.nodes[edge.dst]
        src_label = f"L{src.layer}T{src.token}"
        dst_label = f"L{dst.layer}T{dst.token}"
        x0, y0 = layer_positions[src_label]
        x1, y1 = layer_positions[dst_label]
        color = "#0f766e" if edge.weight >= 0 else "#b91c1c"
        ax.annotate(
            "",
            xy=(x1, y1),
            xytext=(x0, y0),
            arrowprops=dict(arrowstyle="->", lw=2.5, color=color, alpha=0.8),
        )
        xm, ym = (x0 + x1) / 2.0, (y0 + y1) / 2.0
        ax.text(xm, ym + 0.08, f"{edge.weight:+.2f}", color=color, fontsize=8)
    ax.set_xlim(-0.4, 1.6)
    ax.set_ylim(-0.4, 1.4)
    ax.axis("off")
    return ax


def draw_hypothesis(ax, title, mode):
    ax.set_title(title)
    latent = (-0.35, 0.55)
    for label, (x, y) in layer_positions.items():
        ax.scatter(x, y, s=900, color="#f6d365", edgecolor="black", zorder=3)
        ax.text(x, y, label, ha="center", va="center", fontsize=9, weight="bold")
    if mode == "common_cause":
        ax.scatter(*latent, s=700, color="#93c5fd", edgecolor="black", zorder=3)
        ax.text(*latent, "z", ha="center", va="center", fontsize=12, weight="bold")
        arrows = [(latent, layer_positions["L0T0"]), (latent, layer_positions["L0T1"]), (latent, layer_positions["L1T0"])]
    elif mode == "direct_chain":
        arrows = [
            (layer_positions["L0T0"], layer_positions["L0T1"]),
            (layer_positions["L0T0"], layer_positions["L1T0"]),
        ]
    else:
        raise ValueError(mode)
    for start, end in arrows:
        ax.annotate(
            "",
            xy=end,
            xytext=start,
            arrowprops=dict(arrowstyle="->", lw=2.5, color="#374151", alpha=0.85),
        )
    ax.text(0.45, -0.25, "Both stories can fit similar observed co-variation.", ha="center", fontsize=9)
    ax.set_xlim(-0.6, 1.6)
    ax.set_ylim(-0.45, 1.35)
    ax.axis("off")
    return ax


def cosine_kernel(matrix):
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    norms = np.where(norms < 1e-8, 1.0, norms)
    normalized = matrix / norms
    return normalized @ normalized.T


---
## Stage 0 - Start from an observed matrix

Suppose the pipeline has already produced the following tiny normalized effect matrix:

$$
\mathbf{X}_{i,j} = \text{"how much patch node $j$ restores the target behavior on example $i$"}
$$

We will pretend this is our only observation.
The inverse question is: **what shared mechanism could have produced this pattern?**


In [ ]:
X_obs = np.array([
    [ 2.0,  1.8, -1.6,  0.1],
    [ 1.0,  0.9, -0.8, -0.1],
    [ 0.0,  0.1,  0.0,  0.2],
    [-1.0, -0.8,  0.7,  0.0],
    [-2.0, -1.7,  1.5, -0.2],
    [-1.0, -0.9,  0.8,  0.1],
], dtype=np.float32)

example_labels = [f"ex{i}" for i in range(X_obs.shape[0])]
show_matrix(
    "Observed patch-effect matrix X",
    X_obs,
    example_labels,
    node_labels,
    vmin=-2.1,
    vmax=2.1,
)
print("Rows = examples, columns = patch nodes.")
print("By eye, L0T0/L0T1 move together, L1T0 moves in the opposite direction, L1T1 looks mostly noisy.")


A first wrong move would be to rank columns only by average magnitude.
That loses the relational information.

What matters for graph recovery is not just **how large** a node's effects are, but **how its profile varies across examples relative to other nodes**.


In [ ]:
mean_abs = np.mean(np.abs(X_obs), axis=0)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
axes[0].bar(node_labels, mean_abs, color="#2563eb")
axes[0].set_title("Mean absolute effect per node")
axes[0].set_ylabel("mean |effect|")
show_profiles(X_obs, node_labels, ax=axes[1])
show_scatter(X_obs[:, 0], X_obs[:, 1], "L0T0", "L0T1", ax=axes[2])
plt.tight_layout()

print("Mean |effect|:")
for label, value in zip(node_labels, mean_abs):
    print(f"  {label}: {value:.2f}")


---
## Stage 1 - Recover the co-variation structure

The pipeline compresses the inverse problem by looking for stable pairwise structure across examples.
A basic version is the correlation matrix:

$$
\widetilde{\mathbf{X}}_{:,j} = \frac{\mathbf{X}_{:,j} - \mu_j}{\sigma_j},
\qquad
\mathbf{C} = \frac{\widetilde{\mathbf{X}}^\top \widetilde{\mathbf{X}}}{N}
$$

- $C_{uv} > 0$: nodes $u$ and $v$ rise and fall together.
- $C_{uv} < 0$: when one restores more, the other restores less.
- Large $|C_{uv}|$: strong candidate relationship.


In [ ]:
C_obs = correlation_matrix(X_obs)
abs_C = np.abs(C_obs.copy())
np.fill_diagonal(abs_C, 0.0)
si, di = np.unravel_index(np.argmax(abs_C), abs_C.shape)

show_matrix(
    "Correlation matrix C(X)",
    C_obs,
    node_labels,
    node_labels,
    vmin=-1.0,
    vmax=1.0,
)
print(f"Strongest pair by |correlation|: {node_labels[si]} <-> {node_labels[di]} with C = {C_obs[si, di]:+.3f}")


This is the first useful inverse answer.
We still do not know the true hidden story, but we now know which nodes behave as if they belong to the same mechanism.

In PIG, the graph stage adds two constraints:

1. **Direction heuristic**: keep only earlier -> later edges.
2. **Top-k sparsification**: keep only the strongest outgoing edges per node.

That turns a dense similarity matrix into a sparse object we can compare across slices.


In [ ]:
slice_obs = SliceLabel(task="inverse_demo", corruption="shared_signal")
dataset_obs = make_dataset(X_obs, slice_obs)
builder = GraphBuilder(k=2, enforce_direction=True)
graph_obs = builder.build_from_slice(dataset_obs, slice_obs)
adj_obs = graph_obs.get_adjacency_matrix()

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4))
show_matrix(
    "Sparse adjacency after direction + top-k",
    adj_obs,
    node_labels,
    node_labels,
    ax=axes[0],
    vmin=-1.0,
    vmax=1.0,
)
draw_graph(graph_obs, axes[1], "Recovered PatchInfluenceGraph")
plt.tight_layout()

print(f"Nodes: {graph_obs.num_nodes}  Edges: {graph_obs.num_edges}")
for edge in sorted(graph_obs.edges, key=lambda e: -abs(e.weight)):
    src = graph_obs.nodes[edge.src]
    dst = graph_obs.nodes[edge.dst]
    print(f"  L{src.layer}T{src.token} -> L{dst.layer}T{dst.token}   w={edge.weight:+.3f}")


---
## Stage 2 - Why this is still an inverse problem

The graph is a **summary of evidence**, not a proof of the unique hidden mechanism.

Two different stories can fit similar observed co-variation:

- **Common-cause story**: one latent factor $z$ drives several nodes.
- **Direct-chain story**: one observed node directly influences later nodes.

Both can create a matrix where columns co-vary strongly.
So the graph is best understood as a proposal set of candidate relationships.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4))
draw_hypothesis(axes[0], "Hypothesis A: common latent driver", mode="common_cause")
draw_hypothesis(axes[1], "Hypothesis B: direct chain", mode="direct_chain")
plt.tight_layout()

print("Observed matrix alone does not tell us which hidden story is correct.")
print("That is exactly why the later causal-eval stage exists.")


The later causal metrics try to shrink that ambiguity.
For a candidate edge $u -> v$, the pipeline uses quantities like:

$$
I(u \to v) = \frac{\lVert a_v^{\mathrm{patch}(u)} - a_v^{\mathrm{base}} \rVert_2}{\lVert a_v^{\mathrm{clean}} - a_v^{\mathrm{base}} \rVert_2}
$$

$$
M(u \to v) = R(\{u,v\}) - R(\{v\}),
\qquad
\mathrm{nec}(u \to v) = R(\{u\}) - R(\{u, \mathrm{clamp}(v=base)\})
$$

These do not magically solve identifiability, but they are much closer to the causal question than correlation alone.


In [ ]:
real_metrics = {
    "influence": activation_influence_score(
        patched_v=np.array([0.70, 0.80]),
        base_v=np.array([0.00, 0.00]),
        clean_v=np.array([0.80, 1.00]),
    ),
    "mediation": mediation_score(
        restoration_fraction(13.4, base_score=10.0, clean_score=14.0),
        restoration_fraction(11.8, base_score=10.0, clean_score=14.0),
    ),
    "necessity": necessity_score(
        restoration_fraction(13.0, base_score=10.0, clean_score=14.0),
        restoration_fraction(11.4, base_score=10.0, clean_score=14.0),
    ),
}

spurious_metrics = {
    "influence": activation_influence_score(
        patched_v=np.array([0.05, 0.03]),
        base_v=np.array([0.00, 0.00]),
        clean_v=np.array([0.80, 1.00]),
    ),
    "mediation": mediation_score(
        restoration_fraction(11.9, base_score=10.0, clean_score=14.0),
        restoration_fraction(11.8, base_score=10.0, clean_score=14.0),
    ),
    "necessity": necessity_score(
        restoration_fraction(11.9, base_score=10.0, clean_score=14.0),
        restoration_fraction(11.8, base_score=10.0, clean_score=14.0),
    ),
}

metric_names = ["influence", "mediation", "necessity"]
real_values = [real_metrics[name] for name in metric_names]
spurious_values = [spurious_metrics[name] for name in metric_names]

x = np.arange(len(metric_names))
width = 0.35
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.bar(x - width / 2, real_values, width, label="candidate L0T0 -> L0T1", color="#0f766e")
ax.bar(x + width / 2, spurious_values, width, label="candidate L0T1 -> L1T1", color="#b91c1c")
ax.set_xticks(x)
ax.set_xticklabels(metric_names)
ax.set_ylim(0.0, 0.95)
ax.set_title("Toy causal metrics narrow the candidate set")
ax.legend()
plt.tight_layout()

print("Toy causal metrics")
for edge_name, metrics in [("L0T0 -> L0T1", real_metrics), ("L0T1 -> L1T1", spurious_metrics)]:
    print(edge_name)
    for key, value in metrics.items():
        print(f"  {key:>10}: {value:.3f}")


---
## Stage 3 - The final pipeline question is slice-level

Once each slice has its own sparse graph, the pipeline asks a higher-level inverse question:

> Which slices appear to reuse the same circuit pattern?

That is why PIG converts graphs into WL feature vectors and then compares those vectors with kernels.


In [ ]:
X_same_circuit = np.array([
    [ 1.8,  1.6, -1.5,  0.1],
    [ 0.9,  0.8, -0.8,  0.0],
    [ 0.1,  0.0, -0.1,  0.1],
    [-0.9, -0.8,  0.7,  0.0],
    [-1.8, -1.6,  1.4, -0.1],
    [-0.9, -0.8,  0.7,  0.1],
], dtype=np.float32)

X_other_circuit = np.array([
    [ 1.8,  0.1, -0.2,  1.6],
    [ 0.9,  0.0, -0.1,  0.8],
    [ 0.0,  0.1,  0.0,  0.1],
    [-0.9,  0.1,  0.2, -0.8],
    [-1.8, -0.1,  0.2, -1.6],
    [-0.9,  0.0,  0.1, -0.8],
], dtype=np.float32)

slice_same = SliceLabel(task="inverse_demo", corruption="same_circuit")
slice_other = SliceLabel(task="inverse_demo", corruption="other_circuit")

graph_same = builder.build_from_slice(make_dataset(X_same_circuit, slice_same), slice_same)
graph_other = builder.build_from_slice(make_dataset(X_other_circuit, slice_other), slice_other)

graphs = {
    slice_obs: graph_obs,
    slice_same: graph_same,
    slice_other: graph_other,
}
feature_matrix = compute_wl_features(graphs, depth=2)
X_wl = feature_matrix.to_matrix()
K = cosine_kernel(X_wl)
labels = [f"{s.corruption}" for s in feature_matrix.slice_labels]

feature_var = X_wl.var(axis=0)
feature_order = np.argsort(feature_var)[::-1]
active_idx = [idx for idx in feature_order if feature_var[idx] > 0][:10]
if not active_idx:
    active_idx = list(range(min(10, X_wl.shape[1])))
X_wl_view = X_wl[:, active_idx]
feature_labels = [f"f{idx}" for idx in active_idx]

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4))
show_matrix(
    "Most discriminative WL features",
    X_wl_view,
    labels,
    feature_labels,
    ax=axes[0],
    cmap="Blues",
    annotate=True,
)
show_matrix(
    "Cosine similarity over WL features",
    K,
    labels,
    labels,
    ax=axes[1],
    cmap="magma",
    vmin=0.0,
    vmax=1.0,
    annotate=True,
)
plt.tight_layout()

print(f"Full WL matrix shape: {X_wl.shape}")
print(f"Showing the {len(active_idx)} features with non-zero variance across slices: {feature_labels}")
print("Pairwise slice similarities")
for i, li in enumerate(labels):
    for j, lj in enumerate(labels):
        print(f"  {li:>13} vs {lj:<13}: {K[i, j]:.3f}")


## Summary

The notebook started from the *inverse* viewpoint:

1. We observe a patch-effect matrix $\mathbf{X}$.
2. We recover a co-variation matrix $\mathbf{C}$.
3. We sparsify it into a graph $G$.
4. We treat graph edges as hypotheses, not final proofs.
5. We use causal evaluation to test promising edges.
6. We embed graphs so we can compare slices at the circuit-pattern level.

So the pipeline is trying to recover a useful, comparable summary of hidden mechanism structure from intervention data.
That summary is informative, but not uniquely identifiable from correlations alone.
